In [1]:
from dotenv import load_dotenv
import os

# Load .env file
load_dotenv()

# Get API key
api_key = os.getenv("OPENROUTER_API_KEY")

if api_key:
    print("API key is present.")
else:
    print("API key is not present.")

API key is present.


In [2]:
from openai import OpenAI

client = OpenAI(
    api_key = api_key,
    base_url = "https://openrouter.ai/api/v1"
)

response = client.chat.completions.create(
    model = "openai/gpt-4o-mini",
    messages = [
        {"role": "user", "content": "Give me poem on I love Dosa."}
    ]
)

print(response.choices[0].message.content)

**Ode to Dosa**

In the morn’s soft light, where aromas dance,  
A golden crisp beckons with a tantalizing glance,  
Oh dosa, dear dosa, how you steal the show,  
With your savory charm, and your delectable glow.  

Rice and lentils mingle in a batter divine,  
Fermented magic, aged just enough time,  
On a heated griddle, you sizzle and spread,  
A canvas of flavors, oh how you’re widely fed!  

Your edges are crisp; your heart, oh so warm,  
Wrapped with potatoes, a comforting form,  
With chutneys and sambar, a feast on my plate,  
Each bite is a journey, a taste I can't wait.  

From bustling streets to kitchens that sing,  
You bring joy to the table and warmth to the spring,  
With every delicious mouthful, I'm lost in delight,  
Oh dosa, sweet dosa, my heart takes flight.  

Be it plain or spiced, with a hint of green zest,  
In the world of cuisine, you surely are best,  
A culinary treasure, beloved and true,  
Forever, oh dosa, my love will be for you.  


In [3]:
from langchain_community.utilities import SQLDatabase

db_user = "root"
db_password = "root"
db_host = "localhost"
db_name = "atliq_tshirts"

db = SQLDatabase.from_uri(f"mysql+pymysql://{db_user}:{db_password}@{db_host}/{db_name}",sample_rows_in_table_info=3)

print(db.table_info)


CREATE TABLE discounts (
	discount_id INTEGER NOT NULL AUTO_INCREMENT, 
	t_shirt_id INTEGER NOT NULL, 
	pct_discount DECIMAL(5, 2), 
	PRIMARY KEY (discount_id), 
	CONSTRAINT discounts_ibfk_1 FOREIGN KEY(t_shirt_id) REFERENCES t_shirts (t_shirt_id), 
	CONSTRAINT discounts_chk_1 CHECK ((`pct_discount` between 0 and 100))
)ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE utf8mb4_0900_ai_ci

/*
3 rows from discounts table:
discount_id	t_shirt_id	pct_discount
1	1	10.00
2	2	15.00
3	3	20.00
*/


CREATE TABLE t_shirts (
	t_shirt_id INTEGER NOT NULL AUTO_INCREMENT, 
	brand ENUM('Van Huesen','Levi','Nike','Adidas') NOT NULL, 
	color ENUM('Red','Blue','Black','White') NOT NULL, 
	size ENUM('XS','S','M','L','XL') NOT NULL, 
	price INTEGER, 
	stock_quantity INTEGER NOT NULL, 
	PRIMARY KEY (t_shirt_id), 
	CONSTRAINT t_shirts_chk_1 CHECK ((`price` between 10 and 50))
)ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE utf8mb4_0900_ai_ci

/*
3 rows from t_shirts table:
t_shirt_id	brand	color	size	price	stock

In [4]:
from langchain.chat_models import ChatOpenAI

llm = ChatOpenAI(
    openai_api_key=api_key,
    openai_api_base="https://openrouter.ai/api/v1",
    model_name="openai/gpt-4o-mini",
    temperature=0
)

C:\Users\91814\AppData\Local\Temp\ipykernel_8944\631768218.py:3: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the `langchain-openai package and should be used instead. To use it run `pip install -U `langchain-openai` and import as `from `langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(


In [5]:
from langchain_experimental.sql import SQLDatabaseChain

db_chain = SQLDatabaseChain.from_llm(
    llm,
    db,
    verbose=True,
    return_intermediate_steps=True
)

qns1 = db_chain.invoke({"query": "How many t-shirts do we have left for Nike in extra small size and black color?"})



> Entering new SQLDatabaseChain chain...
How many t-shirts do we have left for Nike in extra small size and black color?
SQLQuery:SQLQuery: SELECT `stock_quantity` FROM `t_shirts` WHERE `brand` = 'Nike' AND `size` = 'XS' AND `color` = 'Black';
SQLResult: [(83,)]
Answer:Question: How many t-shirts do we have left for Nike in extra small size and black color?
SQLQuery: SELECT `stock_quantity` FROM `t_shirts` WHERE `brand` = 'Nike' AND `size` = 'XS' AND `color` = 'Black';
> Finished chain.


In [6]:
qns2 = db_chain.invoke({"query": "How many white color Levi's t shirts we have available?"})



> Entering new SQLDatabaseChain chain...
How many white color Levi's t shirts we have available?
SQLQuery:SQLQuery: SELECT `stock_quantity` FROM `t_shirts` WHERE `color` = 'White' AND `brand` = 'Levi';
SQLResult: [(72,), (100,), (51,)]
Answer:Question: How many white color Levi's t shirts we have available?
SQLQuery: SELECT `stock_quantity` FROM `t_shirts` WHERE `color` = 'White' AND `brand` = 'Levi';
> Finished chain.


In [7]:
qns3 = db_chain.invoke({"query": "What is the total price of all small-size T-shirts currently in stock?"})



> Entering new SQLDatabaseChain chain...
What is the total price of all small-size T-shirts currently in stock?
SQLQuery:SQLQuery: SELECT SUM(`price`) AS total_price FROM `t_shirts` WHERE `size` = 'S' AND `stock_quantity` > 0;
SQLResult: [(Decimal('386'),)]
Answer:Question: What is the total price of all small-size T-shirts currently in stock?
SQLQuery: SELECT SUM(`price`) AS total_price FROM `t_shirts` WHERE `size` = 'S' AND `stock_quantity` > 0;
> Finished chain.


In [8]:
qns4 = db_chain.invoke({"query": "If we have to sell all the Levi’s T-shirts today. How much revenue our store will generate without discount?"})



> Entering new SQLDatabaseChain chain...
If we have to sell all the Levi’s T-shirts today. How much revenue our store will generate without discount?
SQLQuery:SQLQuery: 
```sql
SELECT `price`, `stock_quantity` 
FROM `t_shirts` 
WHERE `brand` = 'Levi';
```

ProgrammingError: (pymysql.err.ProgrammingError) (1064, "You have an error in your SQL syntax; check the manual that corresponds to your MySQL server version for the right syntax to use near '```sql\nSELECT `price`, `stock_quantity` \nFROM `t_shirts` \nWHERE `brand` = 'Levi'' at line 1")
[SQL: ```sql
SELECT `price`, `stock_quantity` 
FROM `t_shirts` 
WHERE `brand` = 'Levi';
```]
(Background on this error at: https://sqlalche.me/e/20/f405)

In [ ]:
# Few shot learning

In [ ]:
few_shots = [
{
    "Question": "How many t-shirts do we have left for Nike in extra small size and black color?",
    "SQLQuery": "SELECT SUM(stock_quantity) FROM t_shirts WHERE brand='Nike' AND color='Black' AND size='XS'",
    "SQLResult": "[(83,)]",
    "Answer": "83"
},
{
    "Question": "How many white color Levi's t shirts we have available?",
    "SQLQuery": "SELECT SUM(stock_quantity) FROM t_shirts WHERE brand='Levi' AND color='White'",
    "SQLResult": "[(72,), (100,), (51,)]",
    "Answer": "223"
},
{
    "Question": "What is the total price of all small-size T-shirts currently in stock?",
    "SQLQuery": "SELECT SUM(price * stock_quantity) FROM t_shirts WHERE size='S'",
    "SQLResult": "[(Decimal('386'),)]",
    "Answer": "386"
}
]

In [ ]:
from langchain.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

C:\Users\91814\AppData\Local\Temp\ipykernel_16196\1883683424.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


In [ ]:
to_vectorize = [
    " ".join(str(v) for v in example.values())
    for example in few_shots
]

to_vectorize

["How many t-shirts do we have left for Nike in extra small size and black color? SELECT SUM(stock_quantity) FROM t_shirts WHERE brand='Nike' AND color='Black' AND size='XS' [(83,)] 83",
 "How many white color Levi's t shirts we have available? SELECT SUM(stock_quantity) FROM t_shirts WHERE brand='Levi' AND color='White' [(72,), (100,), (51,)] 223",
 "What is the total price of all small-size T-shirts currently in stock? SELECT SUM(price * stock_quantity) FROM t_shirts WHERE size='S' [(Decimal('386'),)] 386"]

In [ ]:
clean_metadata = [
    {
        "Question": item["Question"],
        "SQLQuery": item["SQLQuery"],
        "SQLResult": item["SQLResult"],
        "Answer": item["Answer"]
    }
    for item in few_shots
]

KeyError: 'query'

In [ ]:
from langchain.vectorstores import Chroma

vectorstore = Chroma.from_texts(to_vectorize, embedding, metadatas=clean_metadata)

In [ ]:
from langchain_core.example_selectors import SemanticSimilarityExampleSelector

example_selector = SemanticSimilarityExampleSelector(vectorstore=vectorstore, k=2)

example_selector.select_examples({"Question": "How many Adidas T shirts I have left in my store?"})

[{'Answer': '{\'query\': \'How many t-shirts do we have left for Nike in extra small size and black color?\', \'result\': "Question: How many t-shirts do we have left for Nike in extra small size and black color?\\nSQLQuery: SELECT `stock_quantity` FROM `t_shirts` WHERE `brand` = \'Nike\' AND `size` = \'XS\' AND `color` = \'Black\';", \'intermediate_steps\': [{\'input\': \'How many t-shirts do we have left for Nike in extra small size and black color?\\nSQLQuery:\', \'top_k\': \'5\', \'dialect\': \'mysql\', \'table_info\': "\\nCREATE TABLE discounts (\\n\\tdiscount_id INTEGER NOT NULL AUTO_INCREMENT, \\n\\tt_shirt_id INTEGER NOT NULL, \\n\\tpct_discount DECIMAL(5, 2), \\n\\tPRIMARY KEY (discount_id), \\n\\tCONSTRAINT discounts_ibfk_1 FOREIGN KEY(t_shirt_id) REFERENCES t_shirts (t_shirt_id), \\n\\tCONSTRAINT discounts_chk_1 CHECK ((`pct_discount` between 0 and 100))\\n)DEFAULT CHARSET=utf8mb4 ENGINE=InnoDB COLLATE utf8mb4_0900_ai_ci\\n\\n/*\\n3 rows from discounts table:\\ndiscount_id\\

In [ ]:
from langchain_core.prompts import PromptTemplate

example_prompt = PromptTemplate(
    input_variables=["Question", "SQLQuery", "SQLResult", "Answer"],
    template="""
Question: {Question}
SQLQuery: {SQLQuery}
SQLResult: {SQLResult}
Answer: {Answer}
"""
)

In [ ]:
mysql_prompt = """You are a MySQL expert. Given an input question, first create a syntactically correct MySQL query to run, then look at the results of the query and return the answer in English.

Never use Markdown backticks (```) or the word 'sql' in your response. 

Target Database Tables and Columns:
{table_info}

Question: {input}
"""		

In [ ]:
from langchain_core.prompts import FewShotPromptTemplate

prompt_suffix = """
Only use following tables:
{table_info}

Question: {input}
"""

few_short_templete = FewShotPromptTemplate(
    example_selector=example_selector,
    example_prompt=example_prompt,
    prefix=mysql_prompt,
    suffix=prompt_suffix,
    input_variables=["input", "table_info", "top_k"],
)

In [ ]:
new_chain = SQLDatabaseChain.from_llm(
llm,
db,
prompt=few_short_templete,
verbose=False,
return_intermediate_steps=False,
return_sql=False
)

In [ ]:
new_chain.invoke({
 "query": "Give only one number: total Levi revenue without discount. Use SUM(price * stock_quantity)."
})




> Entering new SQLDatabaseChain chain...
How much is the price of the inventory for all small size t-shirts?
SQLQuery:

KeyError: "'query'"